# Working with WARC Files

<img src="https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/nlnz-webarchive-public-portal.png" alt="NLNZ Selective Web Archive portal" border="0">

## Overview

This notebook demonstrates how to query and extract web archive data from WARC files using a CDX index. It provides a foundation for working with raw web archive data and accompanying index files.

This notebook uses existing WARC files with pre-generated CDX files, located in an S3 bucket. This S3 bucket has been periodically made public for the use of this notebook in workshops. If the S3 bucket is not longer publically accessible, then this notebook can be reconfigured to point at an alternative endpoint with WARC and CDX data.

Note, some of the code cells contain commented out code for loading CDX and WARC data from local files. To use this, uncomment it, and comment out the equivalent S3 code. This scenario works when running the Jupyter Notebook in a local environment on your device.

### Learning Objectives

1. Understand the roles of WARC and CDX files in web archiving
2. Load and interpret CDX index files
3. Find archived records in WARC files using index metadata
4. Extract textual content from archived web pages

This notebook serves as an introduction to web archive access methods that will be built upon in subsequent notebooks.

## Environment Setup

### Installing Required Python Packages

The following packages are necessary for working with web archives:

In [ ]:
# Colab bootstrap: install the packages required for exploring web archive data in this notebook. You can skip this cell if you are running the notebook in your local environment and have already installed the required packages.
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-toolkit

In [ ]:
# Import the NLNZ Web Archive Toolkit
import wa_nlnz_toolkit as want
import pandas as pd
import datetime
from bs4 import BeautifulSoup

## 1. Working with WARC Files

<img src="https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/warc_cdx_side_by_side_transparent.png" alt="WARC and CDX files" border="0" width="500">

WARC (Web ARChive) files are the standard format for storing web archives. They contain the actual content of archived web pages along with metadata. In this section, we'll explore how to access and extract data from WARC files.

> ***Note***
>
> *This section is using a dataset extracted from the public facing NLNZ web archive. We have made it available via an AWS S3 bucket for the purposes of this workshop.*

In [ ]:
# Define S3 bucket and folder containing the archive data
bucket_name = "ndha-public-data-ap-southeast-2"
folder_prefix = "iPRES-2025"

# List available files in the S3 bucket
want.list_s3_files(bucket_name, folder_prefix)

### Understanding CDX Files

CDX (Capture inDeX) files serve as indexes for WARC files, making it easier to locate specific content. They follow a standardized format described in the [CDX documentation](https://iipc.github.io/warc-specifications/specifications/cdx-format/cdx-2015/).

The standard 11-field CDX format includes:

1. **N**: Normalized/massaged URL
2. **b**: Capture date (timestamp)
3. **a**: Original URL
4. **m**: MIME type of the document
5. **s**: HTTP response code
6. **k**: Content checksum
7. **r**: Redirect URL
8. **M**: Meta tags
9. **S**: Compressed record size
10. **V**: Compressed payload offset
11. **g**: Source WARC filename

Let's load a CDX file and examine its structure:

In [ ]:
# Load a CDX file from S3 into a pandas DataFrame
object_key = (
    "iPRES-2025/sample-data/covid19.govt.nz/2021-08-10_IE75130285/IE75130285.cdx"
)
df = pd.read_csv(f"s3://{bucket_name}/{object_key}", sep=" ", skiprows=1)

# Load a CDX file from your local environment into a pandas Dataframe
# filepath = '/local/path/to/cdx/files/the-cdx-file.cdx'
# df = pd.read_csv(filepath, sep=" ", skiprows=1)

# Assign column names according to the CDX standard
df.columns = ["N", "b", "a", "m", "s", "k", "r", "M", "S", "V", "g"]

# Display the first few rows
df.head(10)

### Extracting Content from WARC Files

Using the information from the CDX file (particularly the filename and offset), we can extract specific content from WARC files:

In [ ]:
# Extract HTML payload from a WARC file in S3 using a specific offset
html_payload = want.extract_payload(
    "s3://ndha-public-data-ap-southeast-2/iPRES-2025/sample-data/covid19.govt.nz/2021-08-10_IE75130285/FL75130287_NLNZ-20210809041626170-00000-22439~kaiwae-z4~8443.warc.gz",
    offset=2593631,
)

# Extract HTML payload from a WARC file in your local environment using a specific offset
# html_payload = want.extract_payload(
#     "/local/path/to/warc/files/the-warc-file.warc.gz",
#     offset=<enter-the-offset>
# )

html_payload

## 2. Processing and Analyzing Web Archive Content

Once we've extracted content from WARC files, we can process and analyze it using various tools. For HTML content, the *BeautifulSoup* Python library is particularly useful for parsing and extracting text.

In [ ]:
# Parse the HTML payload using BeautifulSoup
soup = BeautifulSoup(html_payload, "html.parser")

# Extract text from all paragraph elements
paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]

# Print each paragraph
for para in paragraphs:
    print(para)

### Using the Toolkit's Built-in Functions

The NLNZ Web Archive Toolkit provides convenience functions that combine these steps. For example, `extract_content_html()` extracts text content from HTML payloads:

In [ ]:
# Example of using the toolkit's built-in functions
# Note: This is a reference example and may not run as-is without proper context

# Define a helper function to find WARC files (similar to what we'll use in later notebooks)
bucket_name = "ndha-public-data-ap-southeast-2"
folder_prefix = "iPRES-2025/sample-data/covid19.govt.nz/"
all_files = want.list_s3_files(bucket_name, folder_prefix)


def find_warc_file_path(warc_file):
    """Find the full S3 path for a given WARC filename.

    Args:
        warc_file (str): The WARC filename to search for

    Returns:
        str: Full S3 path if found, None otherwise
    """
    for s3_file in all_files:  # all_files would be defined in actual usage
        if warc_file in s3_file:
            warc_file_path = f"s3://{bucket_name}/{s3_file}"
            return warc_file_path
    return None


# Example usage of extract_content_html (reference only)
warc_file = "FL75130287_NLNZ-20210809041626170-00000-22439~kaiwae-z4~8443.warc.gz"
warc_offset = 2593631

# Extract HTML payload from a WARC file in S3 using a specific offset
html_payload = want.extract_payload(find_warc_file_path(warc_file), warc_offset)

# Extract HTML payload from a WARC file in your local environment using a specific offset
# html_payload = want.extract_payload("/local/path/to/warc/files/the-warc-file.warc.gz", <enter-the-offset>)

content = want.extract_content_html(html_payload)
content

>💡 <strong>Try it:</strong> Using the output from the *Understanding CDX Files* section, try and extract the text content for other pages in the dataset. *Hint - use the `Source WARC filename` and `Compressed payload offset` columns.*

## Conclusion and Next Steps

This notebook has introduced the fundamental methods for accessing and working with WARC and CDX files:

1. **WARC Files** - For accessing archived web content
2. **CDX Files** - For enabling fast, metadata‑driven access to archived web content in collections
2. **Content Extraction** - For processing and analyzing archived web pages

### What's Next?

In the following notebooks, we'll build on these foundations to:

- Explore and analyze web archive data in more depth
- Track changes in websites over time
- Extract and analyze textual content at scale
- Build advanced applications using web archive data

These techniques provide powerful tools for researchers, historians, and data scientists working with web archives.

---

## Hands-On Exercise Solutions



### Citation

**Title:** Working with WARC Files (Jupyter notebook)  
**Author:** Yizhe Zhan, Ben O'Brien  
**Affiliation:** National Library of New Zealand, Wellington  
**Last updated:** 2026‑04‑14  

**Contact:** yizhe.git@gmail.com  
**Repository:** https://github.com/NLNZDigitalPreservation/wa-nlnz-toolkit